# Inferencia difusa cuántica

Veamos cómo implementar el proceso de inferencia difuso mediante Computación Cuántica.

In [20]:
import pennylane as qml
import numpy as np

## Variable de salida

En primer lugar, se prepara un registro de qubits para utilizar como variable de salida. Este registro utiliza  $q = \lceil \text{log}_2 n \rceil + 1$, donde $n$ es el número de etiquetas que tiene la variable de salida. Este registro se inicializa como:

$$\ket{\psi_y} = \ket{1} \otimes \left(\frac{1}{\sqrt{n}}\sum\limits_{i=0}^{n-1} \ket{i} + \sum\limits_{j=n}^{2^q -1} 0\ket{j}\right)$$

La idea de esta inicialización es que, de primeras, todas las salidas son posibles, por lo que tienen igual amplitud. El qubit extra del principio es para indicar aquellas salidas que han sido activadas por el precedente de una regla. De esta forma, para una salida con 3 etiquetas, se tendría la siguiente configuración inicial:

$$\ket{\psi_y} = \ket{1} \otimes \frac{1}{\sqrt{3}}(\ket{00} + \ket{01} + \ket{10})$$

Supongamos que se activa una regla que tiene como consecuente la segunda etiqueta (i.e. $\ket{01}$), el estado pasaría a ser:

$$\ket{\psi_y} = \ket{0} \otimes \frac{1}{\sqrt{3}}\ket{01} + \ket{1} \otimes \frac{1}{\sqrt{3}}(\ket{00} + \ket{10})$$

Estas activaciones no tienen que ser totales, ya que los precedentes de las reglas pueden encontrarse en superposición, pero este comporamiento es el que queremos modelar para implementar la lógica e inferencia difusas. Por lo tanto, cuando midamos el registro de salida, sólo nos quedaremos con las mediciones cuyo primer qubit esté en estado 0, ya que el resto corresponde a las amplitudes de los estados que no se han activado (o el resto de su activación parcial).

#### Ejercicio 1

Implementa una función que inicialize el registro de una variable difusa de salida con tres etiquetas.

In [21]:
def init_quantum_fuzzy_output(values, wires):
    # Inicializa |1> (x) (|00> + |01> + |10>) / sqrt(3) para una salida con 3 etiquetas

    n_labels = 3
    n_state = 2 ** len(wires)
    state = np.zeros(n_state, dtype=complex)

    # Indices para |1>|00>, |1>|01>, |1>|10> en un registro de 3 qubits
    active_indices = [4, 5, 6]
    amp = 1 / np.sqrt(n_labels)
    for idx in active_indices:
        state[idx] = amp

    qml.AmplitudeEmbedding(state, wires=wires, normalize=True)

In [22]:
dev = qml.device("default.qubit", wires=3)


@qml.qnode(dev)
def init_quantum_fuzzy_output_example(values):
    init_quantum_fuzzy_output(values, wires=[0, 1, 2])
    return qml.state()


# check whether the corresponding states have the correct amplitudes
state = init_quantum_fuzzy_output_example(None)
expected = 1 / np.sqrt(3)
indices = [4, 5, 6]
print("Amoplitudes esperadas:", expected)
print("Amplitudes medidas:", [state[i] for i in indices])
print("Norma del estado:", np.linalg.norm(state))

Amoplitudes esperadas: 0.5773502691896258
Amplitudes medidas: [np.complex128(0.5773502691896258+0j), np.complex128(0.5773502691896258+0j), np.complex128(0.5773502691896258+0j)]
Norma del estado: 1.0


## Reglas

Para implementar las reglas que definan el proceso de inferencia, aplicaremos operadores multicontrolados $X$, siendo los qubits de control los registros de las variables de entrada correspondientes y la variable de salida, aplicando las máscaras correspondientes, y actuando sobre el qubit exrta del registro la variable de salida, pasándolo de $\ket{1}$ a $\ket{0}$ según la amplitud de los registros de las variables de entrada. 

De esta forma, se entrelazan los registros de las entradas con los de las salidas, dando un sentido de correlación a ambas partes mediante las reglas.

$$U_{inf}(\ket{x_0} \otimes ... \otimes \ket{x_n} \otimes \ket{\psi_y}) = MCX(\ket{x_0} \otimes ... \otimes \ket{x_n} \otimes (\ket{1} \otimes \frac{1}{\sqrt{n}}(\ket{0...0} + \ket{0...1} + ...))) = $$

$$ = \ket{x_0} \otimes ... \otimes \ket{x_n} \otimes (\ket{0} \otimes \frac{1}{\sqrt{n}}(\ket{0...0} + ... )) + \ket{x_0} \otimes ... \otimes \ket{x_n} \otimes (\ket{1} \otimes \frac{1}{\sqrt{n}}(\ket{0...1} + ...))$$

## *Defuzzificación*

Una vez definido el circuito y ejecutado, se mide el registro de la variable de salida. De los estados que se midan, nos quedamos sólo con aquellos cuyo primer qubit esté en estado $\ket{0}$, ya que son los correspondientes a las activaciones de las reglas (en caso de no haber ninguno, no se ha activado ninguna regla). 

Se realiza el proceso de *defuzzificación* igual que en el proceso clásico, tomando como pesos de activación $w_i$ las amplitudes a partir del número de shots de interés, y las constantes $z_i$ definidas por el propio problema.

#### Ejercicio 2

Implementa un circuito cuántico que modele el proceso de inferencia difusa para el problema del restaurante (Ejericio 1 de ```logica_difusa_2.ipynb```).

In [23]:
# Funciones de pertenencia (mismo ejemplo del restaurante)
def triangular_mf(x, a, b, c):
    return max(min((x - a) / (b - a + 1e-9), (c - x) / (c - b + 1e-9)), 0)


def service_mf(x):
    return {
        "poor": triangular_mf(x, 0, 0, 5),
        "average": triangular_mf(x, 0, 5, 10),
        "excellent": triangular_mf(x, 5, 10, 10),
    }


def food_mf(x):
    return {
        "rancid": triangular_mf(x, 0, 0, 5),
        "average": triangular_mf(x, 0, 5, 10),
        "delicious": triangular_mf(x, 5, 10, 10),
    }


tip_mf = {
    "cheap": 5,
    "average": 10,
    "generous": 30,
}


def encode_fuzzy_input(memberships, wires):
    # Codifica las pertenencias en amplitudes de 2 qubits: |00>, |01>, |10>
    amps = np.array([memberships[0], memberships[1], memberships[2], 0.0], dtype=float)
    if np.allclose(amps, 0):
        amps[0] = 1.0
    qml.AmplitudeEmbedding(amps, wires=wires, normalize=True)


def apply_rule(
    input_wires, input_values, output_label_bits, output_flag_wire, output_label_wires
):
    control_wires = input_wires + output_label_wires
    control_values = input_values + output_label_bits
    qml.MultiControlledX(
        wires=control_wires + [output_flag_wire],
        control_values=control_values,
    )


def defuzzify_quantum_output(probs):
    # Usa solo los estados con flag=0: |000>, |001>, |010>
    label_probs = probs[0:3]
    weights = np.array(
        [tip_mf["cheap"], tip_mf["average"], tip_mf["generous"]], dtype=float
    )
    denom = np.sum(label_probs)
    return float(np.dot(label_probs, weights) / denom) if denom > 0 else 0.0


dev = qml.device("default.qubit", wires=7)


@qml.qnode(dev)
def fuzzy_inference_quantum(service_val, food_val):
    # Registros de entrada (2 qubits por variable)
    service_wires = [0, 1]
    food_wires = [2, 3]
    # Registro de salida (flag + 2 qubits de etiqueta)
    output_wires = [4, 5, 6]
    output_flag = output_wires[0]
    output_label = output_wires[1:]

    service_fuzzy = service_mf(service_val)
    food_fuzzy = food_mf(food_val)

    encode_fuzzy_input(
        [service_fuzzy["poor"], service_fuzzy["average"], service_fuzzy["excellent"]],
        service_wires,
    )
    encode_fuzzy_input(
        [food_fuzzy["rancid"], food_fuzzy["average"], food_fuzzy["delicious"]],
        food_wires,
    )

    # Inicializa la salida con 3 etiquetas
    init_quantum_fuzzy_output(None, wires=output_wires)

    # Reglas (etiquetas: 00=cheap, 01=average, 10=generous)
    cheap_bits = [0, 0]
    average_bits = [0, 1]
    generous_bits = [1, 0]

    # R1: service poor OR food rancid -> cheap
    apply_rule(service_wires, [0, 0], cheap_bits, output_flag, output_label)
    apply_rule(food_wires, [0, 0], cheap_bits, output_flag, output_label)

    # R2: service average -> average
    apply_rule(service_wires, [0, 1], average_bits, output_flag, output_label)

    # R3: service excellent OR food delicious -> generous
    apply_rule(service_wires, [1, 0], generous_bits, output_flag, output_label)
    apply_rule(food_wires, [1, 0], generous_bits, output_flag, output_label)

    return qml.probs(wires=output_wires)


service = 3
food = 8
probs = fuzzy_inference_quantum(service, food)
tip = defuzzify_quantum_output(probs)


print(f"Tip for service={service} and food={food} is {tip}% (expected 16.25%)")

Tip for service=3 and food=8 is 17.27272727272728% (expected 16.25%)
